In [2]:
import pandas as pd
import copy

# 1. Load baseline datasets
node_states_df = pd.read_csv('data/node_states.csv')
edges_df = pd.read_csv('data/temporal_edges.csv')

def run_counterfactual_simulation(target_era, node_id_to_tweak, custom_adjustments):
    """
    Simulates a counterfactual intervention by modifying a node's state 
    in a specific era and propagating the effect to connected neighbors.
    """
    # Filter states for the target era and make a mutable copy
    era_states = node_states_df[node_states_df['Era_Step'] == target_era].copy()
    era_edges = edges_df[edges_df['Era_Step'] == target_era]
    
    # Check if target node exists in this era
    if node_id_to_tweak not in era_states['Node_ID'].values:
        print(f"Node {node_id_to_tweak} not found in era {target_era}.")
        return None

    print(f"--- BASELINE STATE FOR {node_id_to_tweak} ({target_era}) ---")
    baseline_row = era_states[era_states['Node_ID'] == node_id_to_tweak].iloc[0]
    print(baseline_row[['Node_ID', 'Mental_State', 'Stress_Score', 'Palpatine_Proximity']])
    
    # Apply the counterfactual intervention
    for col, new_val in custom_adjustments.items():
        era_states.loc[era_states['Node_ID'] == node_id_to_tweak, col] = new_val

    print(f"\n--- COUNTERFACTUAL INTERVENTION APPLIED ---")
    modified_row = era_states[era_states['Node_ID'] == node_id_to_tweak].iloc[0]
    print(modified_row[['Node_ID', 'Mental_State', 'Stress_Score', 'Palpatine_Proximity']])

    # Simple Graph Diffusion: Propagate stress change to immediate neighbors
    # (Mimicking a basic GNN message-passing layer or social influence model)
    neighbors = era_edges[(era_edges['Source'] == node_id_to_tweak) | (era_edges['Target'] == node_id_to_tweak)]
    neighbor_ids = set(neighbors['Source']).union(set(neighbors['Target'])) - {node_id_to_tweak}

    delta_stress = float(modified_row['Stress_Score']) - float(baseline_row['Stress_Score'])
    
    print(f"\n--- PROPAGATION TO CONNECTED NEIGHBORS ({list(neighbor_ids)}) ---")
    for n_id in neighbor_ids:
        n_idx = era_states[era_states['Node_ID'] == n_id].index
        if not n_idx.empty:
            current_n_stress = float(era_states.loc[n_idx, 'Stress_Score'].values[0])
            # Neighbors absorb a fraction of the change based on edge weight or decay factor
            new_n_stress = max(0.0, min(1.0, current_n_stress + (delta_stress * 0.3)))
            era_states.loc[n_idx, 'Stress_Score'] = new_n_stress
            print(f"Neighbor {n_id} adjusted stress: {current_n_stress:.2f} -> {new_n_stress:.2f}")

    return era_states

In [3]:
# Example: "What if Anakin's Palpatine Proximity was 0.0 and Stress was 0.1 during t11?"
counterfactual_states = run_counterfactual_simulation(
    target_era='t11',
    node_id_to_tweak='Anakin',
    custom_adjustments={
        'Palpatine_Proximity': 0.0,
        'Stress_Score': 0.1,
        'Mental_State': 'Stable'
    }
)

--- BASELINE STATE FOR Anakin (t11) ---
Node_ID                               Anakin
Mental_State           Desperate/Traumatized
Stress_Score                             0.9
Palpatine_Proximity                     0.95
Name: 2, dtype: object

--- COUNTERFACTUAL INTERVENTION APPLIED ---
Node_ID                Anakin
Mental_State           Stable
Stress_Score              0.1
Palpatine_Proximity       0.0
Name: 2, dtype: object

--- PROPAGATION TO CONNECTED NEIGHBORS (['Padmé', 'Mace_Windu', 'Palpatine', 'Obi-Wan']) ---
